In [1]:
# 加载模型
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from transformer_lens import HookedTransformer

# model_path = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
# model_path = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
# model_path = "meta-llama/Llama-3.1-8B-Instruct"
model_path = "model/gemma-2-9b-it"
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_path,torch_dtype=torch.bfloat16,trust_remote_code=True)
# model = HookedTransformer.from_pretrained(model_path, device="cuda")
model.to("cuda")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Gemma2ForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 3584, padding_idx=0)
    (layers): ModuleList(
      (0-41): 42 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear(in_features=3584, out_features=4096, bias=False)
          (k_proj): Linear(in_features=3584, out_features=2048, bias=False)
          (v_proj): Linear(in_features=3584, out_features=2048, bias=False)
          (o_proj): Linear(in_features=4096, out_features=3584, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear(in_features=3584, out_features=14336, bias=False)
          (up_proj): Linear(in_features=3584, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=3584, bias=False)
          (act_fn): PytorchGELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((3584,), eps=1e-06)
        (pre_feedforward_layernorm): G

In [2]:
from sae_lens import SAE
# sae, cfg_dict, sparsity = SAE.from_pretrained(release = "llama_scope_r1_distill", 
#                                               sae_id = "l15r_800m_openr1_math",
#                                               device = "cuda")
sae, cfg_dict, sparsity = SAE.from_pretrained(
    release = "gemma-scope-9b-it-res-canonical",
    sae_id = "layer_9/width_16k/canonical",
    device = "cuda"
)

In [ ]:
import json
#使用构造的数据集
with open("math_cot_v3.jsonl") as f:
    data = [json.loads(line) for line in f]
long_random_samples = [
   
    f'Answer: {data[i]["logical"]}'
    for i in range(len(data))
]
short_random_samples = [
    
    f'Answer: {data[i]["math"]}'
    for i in range(len(data))
]

In [5]:
import numpy as np
A_list= []
for i in range(len(data)):
    
    x = get_layer_act(long_random_samples[i], model,tokenizer)
    len_x = x.size()[0]
    x_sum = torch.sum(x,dim=0)/len_x
    y = get_layer_act(short_random_samples[i], model,tokenizer)
    len_y = y.size()[0]
    y_sum = torch.sum(y,dim=0)/len_y
    f = x_sum.detach().cpu().numpy()-y_sum.detach().cpu().numpy()
    A_list.append(f)
    

    
    

In [6]:
A_list[0]

array([-0.22774659, -0.02101612,  0.4567207 , ..., -0.18869898,
       -1.0097327 , -0.12325665], dtype=float32)

In [7]:
# 计算 AA^T
A = np.column_stack(A_list)
AAT = A @ A.T # 或者 X1 @ X1.T + X2 @ X2.T + X3 @ X3.T

# 特征值分解
eigenvalues, eigenvectors = np.linalg.eigh(AAT)  # 使用 eigh 确保对称性

# 结果验证
Lambda = np.diag(eigenvalues)
U = eigenvectors
print("重构误差:", np.linalg.norm(AAT - U @ Lambda @ U.T))

重构误差: 0.0041338485


In [259]:
eigenvectors[:,-1]

array([-0.00028924, -0.00039788, -0.00290412, ..., -0.00103717,
        0.0017605 , -0.00032302], dtype=float32)

In [8]:
# 保存为.npy文件（二进制格式，高效）
np.save('eigenvectors_gemma.npy', eigenvectors)

# 加载
# loaded_eigenvectors = np.load('eigenvectors.npy')

In [277]:
import torch.nn.functional as F
W_dec = sae.W_dec.cpu()
# 计算 sae.W_dec 每一行与 h 的余弦相似度
similarities = F.cosine_similarity(W_dec, torch.from_numpy(eigenvectors[:,-5]).unsqueeze(0), dim=1)

# 找到前 20 个最大相似度的索引
top_values, top_index = torch.topk(similarities, k=20)

In [278]:
top_index,top_values

(tensor([17993, 30063, 22649, 31784, 22358, 31206, 17917, 29736, 23461, 14684,
         21668, 31552, 28028, 22976,  5427, 19452, 12426, 31443,  7569, 12190]),
 tensor([0.2972, 0.2374, 0.2350, 0.2183, 0.1861, 0.1824, 0.1761, 0.1665, 0.1644,
         0.1606, 0.1606, 0.1603, 0.1577, 0.1576, 0.1572, 0.1559, 0.1533, 0.1525,
         0.1518, 0.1511], grad_fn=<TopkBackward0>))

In [ ]:
import torch.nn.functional as F
W_dec = sae.W_dec.cpu()
# 存储所有符合条件的结果
all_results = []

# 遍历 eigenvectors 的每一列 (i = 1 到 20)
for i in range(1, 51):
    # 获取当前列向量
    h = torch.from_numpy(eigenvectors[:, -i]).unsqueeze(0)  # 转换为 PyTorch 张量并增加维度
    
    # 计算余弦相似度
    similarities = F.cosine_similarity(W_dec, h, dim=1)
    
    # 找到前 20 个最大相似度的索引和值
    top_values, top_indices = torch.topk(similarities, k=20)
    
    # 筛选出相似度大于 0.15 的值及其索引
    valid_indices = top_indices[top_values > 0.15]
    valid_values = top_values[top_values > 0.15]
    
    # 将结果存储为 (值, 索引, i) 的元组列表
    for value, index in zip(valid_values, valid_indices):
        all_results.append((value.item(), index.item(), i))
# 按相似度从大到小排序
all_results.sort(reverse=True, key=lambda x: x[0])

# 提取前 30 个结果
top_10_results = all_results[:30]

# 输出结果
print("Top 10 results:")
for value, index, i in top_10_results:
    print(f"Similarity: {value:.4f}, Index: {index}, Eigenvector Column: {i}")
    print(f"{i}&{value:.2f}&{index}")

Top 10 results:
Similarity: 0.4764, Index: 12085, Eigenvector Column: 2
2&0.48&12085
Similarity: 0.4569, Index: 12085, Eigenvector Column: 1
1&0.46&12085
Similarity: 0.3552, Index: 5012, Eigenvector Column: 1
1&0.36&5012
Similarity: 0.3464, Index: 5012, Eigenvector Column: 2
2&0.35&5012
Similarity: 0.3270, Index: 8927, Eigenvector Column: 7
7&0.33&8927
Similarity: 0.2890, Index: 10568, Eigenvector Column: 9
9&0.29&10568
Similarity: 0.2408, Index: 8927, Eigenvector Column: 8
8&0.24&8927
Similarity: 0.2392, Index: 4342, Eigenvector Column: 6
6&0.24&4342
Similarity: 0.2274, Index: 13526, Eigenvector Column: 22
22&0.23&13526
Similarity: 0.2190, Index: 13724, Eigenvector Column: 2
2&0.22&13724
Similarity: 0.2172, Index: 11268, Eigenvector Column: 5
5&0.22&11268
Similarity: 0.2124, Index: 7675, Eigenvector Column: 20
20&0.21&7675
Similarity: 0.2116, Index: 10678, Eigenvector Column: 2
2&0.21&10678
Similarity: 0.2103, Index: 1534, Eigenvector Column: 1
1&0.21&1534
Similarity: 0.2077, Index: 5

In [50]:
print(len(tokenizer.tokenize(long_random_samples[0])))

50


In [123]:
from IPython.display import display,HTML

import numpy as np
import pdfkit
import os
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.pyplot as plt
import tempfile
def visualize_tokens_with_green_scores(tokens, scores, output_name):
    """
    根据 token 的得分对原文本进行可视化，得分越高颜色越绿。
    
    参数:
        tokens (list): 原文本的 token 列表（如分词后的单词或字符）。
        scores (list): 与 tokens 对应的得分列表。
        
    返回:
        None: 在 Jupyter Notebook 中显示带颜色的文本。
    """
    # 检查输入长度是否一致
    if len(tokens) != len(scores):
        raise ValueError("tokens 和 scores 的长度必须一致！")
    
    # 归一化得分到 [0, 1] 范围
    normalized_scores = (scores - np.min(scores)) / (np.max(scores) - np.min(scores) + 1e-8)
    
    # 创建颜色映射（从无色到深绿色）
    cmap = LinearSegmentedColormap.from_list(
        "green_cmap", 
        [(1, 1, 1, 0), (0, 0.5, 0, 1)]  # RGBA格式：(白色透明) -> (深绿色)
    )
    
    # 构建 HTML 内容
    html_content = "<div style='line-height: 1.5;'>"
    for token, score in zip(tokens, normalized_scores):
        # 获取颜色
        rgb_color = cmap(score)[:3]  # 获取 RGB 值（范围 [0, 1]）
        rgb_color_int = tuple(int(c * 255) for c in rgb_color)  # 转换为 [0, 255]
        color = f"rgb{rgb_color_int}"
        html_content += f"<span style='background-color: {color}; padding: 2px; margin: 1px;'>{token}</span> "
    html_content += "</div>"
    config = pdfkit.configuration(wkhtmltopdf='/usr/bin/wkhtmltopdf')
    pdfkit.from_string(html_content, output_name)
    display(HTML(html_content))

In [124]:
text = '''
 To simplify and write the expression \( \sqrt{\sqrt[3]{\sqrt{\frac{1}{729}}}} \) with a rational denominator, follow these steps: **Step 1:** Simplify the innermost radical. Start with the fraction inside the square root: \[ \frac{1}{729} = \left( \frac{1}{9} \right)^3 \] So, \[ \sqrt{\frac{1}{729}} = \sqrt{\left( \frac{1}{9} \right)^3} = \left( \frac{1}{9} \right)^{\frac{3}{2}} = \frac{1^{\frac{3}{2}}}{9^{\frac{3}{2}}} = \frac{1}{(\sqrt{9})^3} = \frac{1}{27} \] Now we have: \[ \sqrt{\sqrt[3]{\sqrt{\frac{1}{729}}}} = \sqrt{\sqrt[3]{\frac{1}{27}}} \] **Step 2:** Simplify each subsequent radical. Next, take the cube root of \( \frac{1}{27} \): \[ \sqrt[3]{\frac{1}{27}} = \frac{1}{\sqrt[3]{27}} = \frac{1}{3} \] Finally, take the square root of \( \frac{1}{3} \): \[ \sqrt{\frac{1}{3}} = \frac{\sqrt{1}}{\sqrt{3}} = \frac{1}{\sqrt{3}} \] **Step 3:** Rationalize the denominator if necessary. The simplified form is already without a rational denominator. However, to fully rationalize it: Multiply numerator and denominator by \( \sqrt{3} \): \[ \frac{1}{\sqrt{3}} \times \frac{\sqrt{3}}{\sqrt{3}} = \frac{\sqrt{3}}{3} \] Thus, the expression simplifies to \( \frac{\sqrt{3}}{3} \). **Final Answer:** \[ \boxed{\dfrac{\sqrt{3}}{3}} \]'''

In [125]:
feature_idx =13526
x = get_layer_act(text, model,tokenizer)

# 生成与 eigenvectors[:,-1] 相同维度的随机向量
random_vector = torch.randn(eigenvectors.shape[0])  # 标准正态分布采样
# 归一化为单位向量
unit_random_vector = random_vector / torch.norm(random_vector, p=2)
# 乘以10（保持与原代码相同的尺度）
result = 10 * unit_random_vector

modify_x = x +10*torch.from_numpy(eigenvectors[:,-22])
# modify_x = x + result
tokens = tokenizer.tokenize(text)
long_acts = sae.encode(x.to("cuda").to(torch.float32))
modify_acts = sae.encode(modify_x.to("cuda").to(torch.float32))
new_tokens = [ele.strip("Ġ").replace("Č","").replace("Ċ","").replace("▁","").replace("Ĉ","") for ele in tokens]
scores = long_acts[1:,feature_idx].detach().cpu().numpy()
modify_scores = modify_acts[1:,feature_idx].detach().cpu().numpy()
# print(scores)
# print(modify_scores)
# save_html_as_pdf(new_tokens,modify_scores,"example2.pdf")
visualize_tokens_with_green_scores(new_tokens, scores, f"gemma_orign_22_{feature_idx}.pdf") 
visualize_tokens_with_green_scores(new_tokens, modify_scores,f"gemma_modify_22_{feature_idx}.pdf") 